# Uji Skenario — Extractive Summarization
Lima skenario pengujian untuk pendekatan extractive (TF-IDF, TextRank, BERT),
memakai hasil ringkasan yang sudah ada (`hasil_summary.csv`).

**Urutan dioptimalkan:** Skenario 4 (cari model terbaik) dijalankan lebih dulu,
karena skenario lain memakai info "model extractive terbaik".

| Skenario | Metrik | Data |
|----------|--------|------|
| 4. Tiga model | ROUGE + BERTScore | Penuh / sample BERTScore |
| 1. Panjang artikel | ROUGE | Penuh (re-use) |
| 2. Kategori | ROUGE | Penuh (re-use) |
| 3. Preprocessing | ROUGE | Sample 500 (generate ulang) |
| 5. vs Abstractive | ROUGE + BERTScore | Dataset sama |

---
## Persiapan

### Setup & instalasi

In [1]:
# Install dependensi (jalankan sekali)
!pip install -q rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 93.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26

### Load data
`hasil_summary.csv` tidak menyimpan `article_text` (dibutuhkan Skenario 3),
jadi diambil dari `hasil_gabungan_abstractive.csv` lewat merge by `global_id`.

In [2]:
import re
import numpy as np
import pandas as pd

# CSV hasil ringkasan extractive (TF-IDF, TextRank, BERT sudah dihasilkan)
PATH_EXTRACTIVE = '/kaggle/input/datasets/nazhifberlian/datauji/hasil_summary (1).csv'

# CSV ini dipakai HANYA untuk mengambil kolom article_text (teks asli),
# karena hasil_summary.csv tidak menyimpannya. Dibutuhkan untuk Skenario 3.
PATH_ARTIKEL    = '/kaggle/input/datasets/nazhifberlian/datauji/hasil_gabungan_abstractive.csv'

df = pd.read_csv(PATH_EXTRACTIVE, encoding='utf-8-sig')
print(f'Extractive : {len(df)} baris, kolom: {list(df.columns)}')

# Ambil article_text via merge by global_id (kedua file punya ID yang sama)
df_artikel = pd.read_csv(PATH_ARTIKEL, encoding='utf-8-sig')
if 'article_text' in df_artikel.columns:
    df = df.merge(df_artikel[['global_id', 'article_text']], on='global_id', how='left')
    print(f'article_text berhasil digabung. Total kolom: {list(df.columns)}')
else:
    print('[!] article_text tidak ditemukan — Skenario 3 perlu penyesuaian sumber teks.')

Extractive : 12644 baris, kolom: ['global_id', 'title', 'category', 'summary_tfidf', 'summary_textrank', 'summary_bert', 'lead_paragraph', 'body_word_count']
article_text berhasil digabung. Total kolom: ['global_id', 'title', 'category', 'summary_tfidf', 'summary_textrank', 'summary_bert', 'lead_paragraph', 'body_word_count', 'article_text']


### Cek ringkasan kosong/NaN (cegah crash saat ROUGE)

In [3]:
# Cek ringkasan kosong/NaN sebelum evaluasi agar ROUGE tidak crash.
SUMMARY_COLS = ['summary_tfidf', 'summary_textrank', 'summary_bert']

for col in SUMMARY_COLS:
    n_kosong = df[col].isna().sum() + (df[col].astype(str).str.strip() == '').sum()
    print(f'{col:<20}: {n_kosong} baris kosong/NaN')

df = df.dropna(subset=SUMMARY_COLS).reset_index(drop=True)
print(f'\nDataset setelah pembersihan: {len(df)} baris')
print(f'\nDistribusi kategori:')
print(df['category'].value_counts().to_string())

summary_tfidf       : 0 baris kosong/NaN
summary_textrank    : 0 baris kosong/NaN
summary_bert        : 0 baris kosong/NaN

Dataset setelah pembersihan: 12644 baris

Distribusi kategori:
category
sejarah     1976
arts        1965
artis       1947
kuliner     1944
tech        1716
biografi    1679
sains       1417


### Setup ROUGE scorer

In [4]:
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

def rouge_avg(pred_series, ref_series):
    # Rata-rata F-measure ROUGE-1/2/L atas dua kolom teks.
    r1, r2, rl = [], [], []
    for pred, ref in zip(pred_series, ref_series):
        s = scorer.score(str(ref), str(pred))
        r1.append(s['rouge1'].fmeasure)
        r2.append(s['rouge2'].fmeasure)
        rl.append(s['rougeL'].fmeasure)
    return np.mean(r1), np.mean(r2), np.mean(rl)

# Ketiga metode extractive yang dibandingkan
models = [
    ('TF-IDF',   'summary_tfidf'),
    ('TextRank', 'summary_textrank'),
    ('BERT',     'summary_bert'),
]

print('ROUGE scorer siap.')

ROUGE scorer siap.


---
## Skenario 4 — Perbandingan Tiga Model Extractive
Dijalankan pertama untuk menentukan model terbaik yang dipakai skenario lain.

In [5]:
# ============================================================
# UJI SKENARIO 4 — Perbandingan Tiga Model Extractive
# ============================================================
# Dijalankan PERTAMA karena Skenario 1, 2, 3, 5 membutuhkan info
# "model extractive terbaik". Bandingkan TF-IDF, TextRank, BERT
# pada seluruh dataset dengan ROUGE.

rows = []
for name, col in models:
    r1, r2, rl = rouge_avg(df[col], df['lead_paragraph'])
    rows.append({'model': name, 'rouge1': r1, 'rouge2': r2, 'rougeL': rl})

hasil_model = pd.DataFrame(rows)
print('Perbandingan tiga model extractive (seluruh dataset):')
print('=' * 55)
print(hasil_model.round(4).to_string(index=False))

# Tentukan model terbaik (ROUGE-L) untuk dipakai skenario lain
best_row = hasil_model.loc[hasil_model['rougeL'].idxmax()]
MODEL_TERBAIK_NAMA = best_row['model']
MODEL_TERBAIK_KOL = dict(models)[MODEL_TERBAIK_NAMA]
print(f'\nModel extractive terbaik (ROUGE-L): {MODEL_TERBAIK_NAMA} ({MODEL_TERBAIK_KOL})')

Perbandingan tiga model extractive (seluruh dataset):
   model  rouge1  rouge2  rougeL
  TF-IDF  0.2069  0.0460  0.1198
TextRank  0.1984  0.0482  0.1245
    BERT  0.1892  0.0432  0.1186

Model extractive terbaik (ROUGE-L): TextRank (summary_textrank)


---
## Skenario 1 — Pengaruh Panjang Artikel
Kelompok: Pendek (<300), Sedang (300-500), Panjang (>500).

In [6]:
# ============================================================
# UJI SKENARIO 1 — Pengaruh Panjang Artikel terhadap Kualitas
# ============================================================
# Kelompokkan ringkasan yang SUDAH ada berdasarkan panjang artikel
# (body_word_count), hitung ROUGE-L per kelompok. Pakai model terbaik
# dari Skenario 4. Tidak perlu generate ulang.

def length_group(wc):
    if wc < 300:
        return '1_Pendek (<300)'
    elif wc <= 500:
        return '2_Sedang (300-500)'
    else:
        return '3_Panjang (>500)'

df['length_group'] = df['body_word_count'].apply(length_group)

rows = []
for grp in sorted(df['length_group'].unique()):
    sub = df[df['length_group'] == grp]
    r1, r2, rl = rouge_avg(sub[MODEL_TERBAIK_KOL], sub['lead_paragraph'])
    rows.append({'kelompok': grp, 'jumlah': len(sub),
                 'rouge1': r1, 'rougeL': rl})

hasil_panjang = pd.DataFrame(rows)
print(f'Pengaruh panjang artikel terhadap ROUGE ({MODEL_TERBAIK_NAMA}):')
print('=' * 55)
print(hasil_panjang.round(4).to_string(index=False))

Pengaruh panjang artikel terhadap ROUGE (TextRank):
          kelompok  jumlah  rouge1  rougeL
   1_Pendek (<300)    6270  0.1834  0.1203
2_Sedang (300-500)    2902  0.2025  0.1284
  3_Panjang (>500)    3472  0.2220  0.1287


---
## Skenario 2 — Pengaruh Kategori
Evaluasi per tujuh kategori artikel.

In [7]:
# ============================================================
# UJI SKENARIO 2 — Pengaruh Kategori terhadap Kualitas Ringkasan
# ============================================================
# Hitung ROUGE per kategori (7 kategori) memakai model terbaik.

rows = []
for kat in sorted(df['category'].unique()):
    sub = df[df['category'] == kat]
    r1, r2, rl = rouge_avg(sub[MODEL_TERBAIK_KOL], sub['lead_paragraph'])
    rows.append({'kategori': kat, 'jumlah': len(sub),
                 'rouge1': r1, 'rougeL': rl})

hasil_kategori = pd.DataFrame(rows).sort_values('rougeL', ascending=False)
print(f'Pengaruh kategori terhadap ROUGE ({MODEL_TERBAIK_NAMA}):')
print('=' * 55)
print(hasil_kategori.round(4).to_string(index=False))
print(f'\nKategori terbaik : {hasil_kategori.iloc[0]["kategori"]} '
      f'(ROUGE-L {hasil_kategori.iloc[0]["rougeL"]:.4f})')
print(f'Kategori terendah: {hasil_kategori.iloc[-1]["kategori"]} '
      f'(ROUGE-L {hasil_kategori.iloc[-1]["rougeL"]:.4f})')

Pengaruh kategori terhadap ROUGE (TextRank):
kategori  jumlah  rouge1  rougeL
 sejarah    1976  0.2191  0.1348
    tech    1716  0.2109  0.1301
 kuliner    1944  0.2009  0.1287
   sains    1417  0.1971  0.1218
    arts    1965  0.1868  0.1199
   artis    1947  0.1907  0.1180
biografi    1679  0.1817  0.1170

Kategori terbaik : sejarah (ROUGE-L 0.1348)
Kategori terendah: biografi (ROUGE-L 0.1170)


---
## Skenario 3 — Pengaruh Preprocessing
Bandingkan dengan vs tanpa preprocessing. Generate ulang pada sample N=500.

### 3a. Definisi fungsi extractive untuk generate ulang

In [8]:
# ============================================================
# UJI SKENARIO 3 — Pengaruh Preprocessing terhadap Kualitas
# ============================================================
# Bandingkan ringkasan dari teks BERSIH (preprocessed) vs teks MENTAH
# (article_text apa adanya), memakai metode extractive terbaik.
# Generate ulang pada SAMPLE acak (N=500) agar ringan & valid.
#
# Extractive jauh lebih ringan dari abstractive (tanpa inferensi model
# generatif), jadi aman di Kaggle tanpa memory error.

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import networkx as nx

STOPWORDS_ID = set("""
yang di ke dari dan atau pada untuk dengan adalah ini itu dalam tidak akan
sebagai juga oleh karena sudah dapat telah agar bisa saat para suatu setelah
hingga maupun sebuah merupakan namun yaitu serta secara hanya masih lebih tersebut
""".split())

def split_sentences(text):
    sents = sent_tokenize(str(text))
    return [s.strip() for s in sents if len(s.split()) >= 4]

# Preprocessing: pembersihan teks (versi WITH preprocessing)
def clean_text(text):
    text = str(text)
    text = re.sub(r'\[\d+\]', '', text)
    text = re.sub(r'={2,}.*?={2,}', '', text)
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'\{\{.*?\}\}', '', text, flags=re.DOTALL)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# Metode extractive terbaik diasumsikan TextRank (sesuaikan bila beda).
# Kalau model terbaik dari Skenario 4 adalah TF-IDF/BERT, ganti fungsi ini.
def summarize_textrank(text, n_sentences=3):
    sentences = split_sentences(text)
    if len(sentences) <= n_sentences:
        return ' '.join(sentences)
    vec = TfidfVectorizer(stop_words=list(STOPWORDS_ID), lowercase=True)
    m = vec.fit_transform(sentences)
    sim = cosine_similarity(m)
    np.fill_diagonal(sim, 0)
    graph = nx.from_numpy_array(sim)
    scores = nx.pagerank(graph, max_iter=100)
    ranked = sorted(scores, key=scores.get, reverse=True)
    top_idx = sorted(ranked[:n_sentences])
    return ' '.join(sentences[i] for i in top_idx)

print('Fungsi extractive untuk Skenario 3 siap.')

Fungsi extractive untuk Skenario 3 siap.


### 3b. Jalankan & bandingkan

In [9]:
from tqdm import tqdm

SAMPLE_SIZE_PRE = 500
sample3 = df.dropna(subset=['article_text']).sample(
    n=min(SAMPLE_SIZE_PRE, len(df)), random_state=42).reset_index(drop=True)

N_KALIMAT = 3  # jumlah kalimat ringkasan tetap untuk perbandingan adil

# Versi A: DENGAN preprocessing (bersihkan dulu, baru ringkas)
tqdm.pandas(desc='Dengan preprocessing')
sample3['sum_clean'] = sample3['article_text'].progress_apply(
    lambda x: summarize_textrank(clean_text(x), N_KALIMAT))

# Versi B: TANPA preprocessing (teks mentah apa adanya)
tqdm.pandas(desc='Tanpa preprocessing')
sample3['sum_raw'] = sample3['article_text'].progress_apply(
    lambda x: summarize_textrank(x, N_KALIMAT))

# Bandingkan ROUGE kedua versi pada sampel yang sama
clean_scores = rouge_avg(sample3['sum_clean'], sample3['lead_paragraph'])
raw_scores   = rouge_avg(sample3['sum_raw'], sample3['lead_paragraph'])

print(f'Pengaruh preprocessing (sample N={len(sample3)}, seed=42):')
print('=' * 55)
print(f'{"":22} | {"ROUGE-1":>8} | {"ROUGE-2":>8} | {"ROUGE-L":>8}')
print('-' * 55)
print(f'{"Dengan preprocessing":22} | {clean_scores[0]:>8.4f} | {clean_scores[1]:>8.4f} | {clean_scores[2]:>8.4f}')
print(f'{"Tanpa preprocessing":22} | {raw_scores[0]:>8.4f} | {raw_scores[1]:>8.4f} | {raw_scores[2]:>8.4f}')
print('-' * 55)
selisih = clean_scores[2] - raw_scores[2]
print(f'Selisih ROUGE-L: {selisih:+.4f} ({selisih / raw_scores[2] * 100:+.1f}%)')

Tanpa preprocessing: 100%|██████████| 500/500 [00:02<00:00, 200.93it/s]


Pengaruh preprocessing (sample N=500, seed=42):
                       |  ROUGE-1 |  ROUGE-2 |  ROUGE-L
-------------------------------------------------------
Dengan preprocessing   |   0.2177 |   0.0526 |   0.1317
Tanpa preprocessing    |   0.2177 |   0.0526 |   0.1317
-------------------------------------------------------
Selisih ROUGE-L: -0.0000 (-0.0%)


---
## Skenario 4 (lanjutan) — BERTScore Tiga Model
Metrik berbasis makna untuk melengkapi ROUGE.

In [10]:
# ============================================================
# UJI SKENARIO 4 (lanjutan) — BERTScore Tiga Model Extractive
# ============================================================
# BERTScore melengkapi ROUGE: mengukur kemiripan MAKNA, bukan overlap kata.
# Berat (load model + encoding), jadi dihitung pada sampel acak N=500.
# Pakai sampel sama (seed=42) agar ketiga model dibandingkan adil.
#
# CATATAN BIAS: BERTScore pakai model multilingual default (lang='id'),
# BERBEDA dari MiniLM yang dipakai metode extractive BERT — menghindari
# keuntungan tidak adil saat menilai ringkasan BERT.
from bert_score import score as bertscore

SAMPLE_SIZE_BS = 500
sample_bs = df.sample(n=min(SAMPLE_SIZE_BS, len(df)), random_state=42).reset_index(drop=True)
refs_bs = sample_bs['lead_paragraph'].astype(str).tolist()

def hitung_bertscore(pred_col):
    cands = sample_bs[pred_col].astype(str).tolist()
    P, R, F1 = bertscore(cands, refs_bs, lang='id', verbose=False)
    return P.mean().item(), R.mean().item(), F1.mean().item()

print(f'Menghitung BERTScore pada {len(sample_bs)} artikel sampel...\n')
print(f'{"Model":<10} | {"Precision":>9} | {"Recall":>9} | {"F1":>9}')
print('-' * 48)
hasil_bertscore = {}
for name, col in models:
    p, r, f1 = hitung_bertscore(col)
    hasil_bertscore[name] = (p, r, f1)
    print(f'{name:<10} | {p:>9.4f} | {r:>9.4f} | {f1:>9.4f}')

best_bs = max(hasil_bertscore, key=lambda k: hasil_bertscore[k][2])
print(f'\nTerbaik menurut BERTScore-F1: {best_bs}')

Menghitung BERTScore pada 500 artikel sampel...

Model      | Precision |    Recall |        F1
------------------------------------------------


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextRank   |    0.6580 |    0.6325 |    0.6443


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BERT       |    0.6603 |    0.6323 |    0.6454

Terbaik menurut BERTScore-F1: BERT


---
## Skenario 5 — Extractive Terbaik vs Abstractive Terbaik
Model abstractive terbaik: T5 (dari hasil evaluasi temanmu).

In [11]:
# ============================================================
# UJI SKENARIO 5 — Extractive Terbaik vs Abstractive Terbaik
# ============================================================
# Membandingkan model terbaik kedua pendekatan pada DATASET YANG SAMA.
#
# PENTING (perbaikan keadilan): agar perbandingan valid, skor extractive
# dihitung ULANG di sini pada dataset yang sama, BUKAN menyalin angka dari
# notebook lain dengan sample berbeda. Skor abstractive diambil dari hasil
# evaluasi resmi (lihat di bawah).

# --- Model abstractive terbaik: T5 (dari hasil evaluasi temanmu) ---
# Sumber: perbandingan tiga model abstractive (T5 terbaik di ROUGE-L & BERTScore)
SKOR_ABSTRACTIVE_T5 = {
    'rouge1':      0.1729,
    'rouge2':      0.0416,
    'rougeL':      0.1172,
    'bertscore_f1': 0.6605,
}

# --- Model extractive terbaik: hitung ulang pada seluruh dataset ---
ext_r1, ext_r2, ext_rl = rouge_avg(df[MODEL_TERBAIK_KOL], df['lead_paragraph'])
ext_bs_f1 = hasil_bertscore[MODEL_TERBAIK_NAMA][2]

print('Perbandingan Extractive Terbaik vs Abstractive Terbaik:')
print('=' * 60)
print(f'{"Pendekatan":<24} | {"ROUGE-1":>8} | {"ROUGE-L":>8} | {"BERTScore":>9}')
print('-' * 60)
print(f'{f"Extractive ({MODEL_TERBAIK_NAMA})":<24} | {ext_r1:>8.4f} | {ext_rl:>8.4f} | {ext_bs_f1:>9.4f}')
print(f'{"Abstractive (T5)":<24} | {SKOR_ABSTRACTIVE_T5["rouge1"]:>8.4f} | '
      f'{SKOR_ABSTRACTIVE_T5["rougeL"]:>8.4f} | {SKOR_ABSTRACTIVE_T5["bertscore_f1"]:>9.4f}')
print('=' * 60)

# Kesimpulan per metrik
print('\nKesimpulan:')
menang_rouge = MODEL_TERBAIK_NAMA if ext_rl > SKOR_ABSTRACTIVE_T5['rougeL'] else 'T5'
menang_bs = MODEL_TERBAIK_NAMA if ext_bs_f1 > SKOR_ABSTRACTIVE_T5['bertscore_f1'] else 'T5'
print(f'  Menurut ROUGE-L   : {menang_rouge} unggul')
print(f'  Menurut BERTScore : {menang_bs} unggul')
if menang_rouge != menang_bs:
    print('  -> Pemenang BERBEDA antar metrik: extractive unggul di overlap kata (ROUGE),')
    print('     abstractive lebih kompetitif di kemiripan makna (BERTScore).')

# CATATAN: skor abstractive BERTScore (0.6605) dihitung pada sample N=500 di
# notebook abstractive. Untuk perbandingan paling ketat, pastikan sample yang
# sama persis. Skor ROUGE abstractive dihitung pada dataset penuh.

Perbandingan Extractive Terbaik vs Abstractive Terbaik:
Pendekatan               |  ROUGE-1 |  ROUGE-L | BERTScore
------------------------------------------------------------
Extractive (TextRank)    |   0.1984 |   0.1245 |    0.6443
Abstractive (T5)         |   0.1729 |   0.1172 |    0.6605

Kesimpulan:
  Menurut ROUGE-L   : TextRank unggul
  Menurut BERTScore : T5 unggul
  -> Pemenang BERBEDA antar metrik: extractive unggul di overlap kata (ROUGE),
     abstractive lebih kompetitif di kemiripan makna (BERTScore).
